This notebook is designed to determine the base window for neutron detection.
We use data from a reference experiment with sufficient neutron signal (currently ID-419).
The PSD/Energy data is then put into a 2D histogram, with energy cuts approximately every 15 keVee, and X total PSD cuts.
This 2D histogram is split using the energy cuts into a series of 1D histogram "energy" slices.
Each slice is modeled on a bimodal distribution of PSD vs. count, with the lower PSD Gaussian being for gamma ray events, and the higher PSD Gaussian being for neutrons.
Using these fit values, the FOM value is also calculated as abs(mu_n-mu_g)/(2.56*(sigma_n+sigma_g))
These values are used to define the neutron window as follows:

- Left boundary: Find energy A where FOM passes 1.27
- Right boundary: E = 688 keVee (Compton scattering edge, adjusted by detector energy resolution)
- Bottom boundary: For each slice, get point where x = slice energy midpoint, y = mu_n - 3 * sigma_n, connect points
- Top boundary: For each slice, get point where x = slice energy midpoint, y = mu_n + 3 * sigma_n, connect points

## Initialization

In [ ]:
# Importing needed code

from math import exp
from pathlib import Path
from typing import Callable, TypeVar, Any
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import deconvolve, convolve

from data_processing.arc_paths import (
    get_parq_root, get_exp_root, INPUT_DATA_FOLDER, get_report_root
)
from data_processing.dataframe_validation import DetectorDataframeColumn
from data_processing.experiment_data_keys import ExperimentDataKey
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    find_failed_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.figure_of_merit import gaussian
from data_processing.processing.neutron_classification import classify
from data_processing.processing.neutron_window_generation import (
    generate_nasa_neutron_window,
    generate_n_distro_neutron_window
)
from data_processing.reporting.plotting import plot_classification
from data_processing.helpers.stop_jupyter import stop

In [ ]:
# Constants
SMOOTHING_WINDOW_SIZE = 5

In [ ]:
T = TypeVar('T')


def get_input_with_default(
    prompt: str, default: int, converter: Callable[[str], T]
) -> int:
    raw_value = input(prompt)
    try:
        value = converter(raw_value)
    except ValueError:
        value = default
    return value

In [ ]:
# def get_window_size(data_type: str) -> int:
#     prompt = (f"Enter smoothing window size for {data_type}, "
#               + "or press Enter for default (5):")
#     return get_input_with_default(prompt, SMOOTHING_WINDOW_SIZE, int)


# def load_non_neutron_data(exp_name, file_names):
#     if not isinstance(file_names, list):
#         file_names = [file_names]
#     for file_name in file_names:
#         file_path = get_exp_root(exp_name) / file_name
#         if file_path.is_file():
#             df = pd.read_csv(file_path)
#             return df
#     return None


# def smooth_non_neutron_data(
#     df, raw_data_col_name, smoothed_data_col_name, window_size
# ):
#     df[smoothed_data_col_name] = df[raw_data_col_name].rolling(
#         window=window_size, min_periods=1).mean()
#     return df


# def localize_time(df, time_column):
#     local_tz = 'America/Vancouver'
#     naive_time = pd.to_datetime(df[time_column])
#     try:
#         localized_time = naive_time.dt.tz_localize(local_tz)
#     except TypeError:
#         localized_time = naive_time.dt.tz_convert(local_tz)
#     df[time_column] = localized_time
#     return df

In [ ]:
# def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
#     start_time = time_bins[0]
#     df = get_time_cut(df, 'Time', time_bins)

#     binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
#         .agg(['mean', 'std']) \
#         .copy()
#     binned_df.columns = selected_cols
#     binned_df['Bin midpoint'] = binned_df.index.to_series() \
#         .apply(lambda x: x.mid)
#     binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

#     return binned_df

In [ ]:
# def bin_midpoint_time_to_seconds(df, start_time):
#     zeroed_midpoint = pd.to_datetime(df["Bin midpoint"]) - start_time
#     df['Bin time (s)'] = zeroed_midpoint.dt.total_seconds()
#     return df

In [ ]:
# def get_time_cut(df, time_tag_col, time_bins):
#     timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
#     df['Time Bin'] = timetag_cut
#     return df

In [ ]:
# def get_endings_with_modifications(endings, string_method):
#     modded_endings = [
#         *endings,
#         *[getattr(ending, string_method)() for ending in endings]
#     ]
#     return modded_endings


# def get_possible_names(exp_name, endings):
#     possible_names = [f"{exp_name}{ending}.csv"
#                       for ending in possible_filename_endings]
#     return possible_names


# def get_filenames_with_hyphenless_ids(exp_name, filenames):
#     hyphenless = exp_name.replace('-', '')
#     modded_filenames = [
#         *filenames,
#         *[filename.replace(exp_name, hyphenless) for filename in filenames]
#     ]
#     return modded_filenames


# def get_full_possible_names_set(exp_name, endings):
#     possible_names = get_possible_names(exp_name, endings)
#     full_poss_names_set = get_filenames_with_hyphenless_ids(
#         exp_name, possible_names)
#     return full_poss_names_set

In [ ]:
# def find_failed_slices(
#     df: pd.DataFrame,
#     nan_total_threshold: int = 5,  # max bad slice fits total
#     nan_window_threshold: int = 4,  # max bad slice fits in a "window"
#     nan_rolling_window: int = 7  # window size
# ) -> tuple[pd.DataFrame, np.ndarray | None]:
#     bad_slice_indexes = None
#     nan_rows = df.isna().any(axis=1)
#     nan_rows = nan_rows[nan_rows]
#     if nan_rows.shape[0] > 0:
#         nan_indexes = np.where(nan_rows)[0]
#         total_nan_rows = len(nan_indexes)
#         rolling_nan_count = nan_rows.rolling(window=nan_rolling_window) \
#             .sum() \
#             .max()

#         print(f"Fit issues in {experiment_id}")
#         print(f"Fit failed on following slice indexes: {nan_indexes}")

#         if (total_nan_rows > nan_total_threshold
#                 or rolling_nan_count > nan_window_threshold):
#             print(f"Experiment {experiment_id} could not be classified")
#             print(f"Total failed slices: {total_nan_rows}")
#             print(
#                 f"Max failed slices in a {nan_rolling_window} slice window:"+
#                 f" {rolling_nan_count}"
#             )

#             df = df.dropna().copy()
#             bad_slice_indexes = nan_indexes

#         # filter out all nan rows from df
#         df = df.dropna().copy()
#         # continue as normal to try fitting with bad rows ignored
#     return df, bad_slice_indexes

## Experiment ID Input

In [ ]:
experiment_id = "ID-419"
# experiment_id = "ID-375"
# experiment_id = "ID-463"
id_valid = get_parq_root(experiment_id).is_dir()
if id_valid:
    print(f"Experiment {experiment_id} found")
else:
    print(f"Experiment {experiment_id} cannot be found")
    stop()
# done = False
# experiment_ids = ["ID-213"]
# while not done:
#     ids_valid = []
#     for exp_id in experiment_ids:
#         id_valid = get_parq_root(exp_id).is_dir()
#         ids_valid.append(id_valid)
#         if not id_valid:
#             print(f"Experiment {exp_id} cannot be found")

#     done = all(ids_valid)
#     if not done:
#         print("Invalid experiment IDs, please fix")
#         experiment_ids = []
#     else:
#         print("All experiment IDs are valid")

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
exp_data: dict[str, Any] = {ExperimentDataKey.UNCLASSIFIED: load_psd(experiment_id)}

In [ ]:
# Express timetags in hours elapsed
exp_data[ExperimentDataKey.UNCLASSIFIED] = calculate_timetag_hours(exp_data[ExperimentDataKey.UNCLASSIFIED])

In [ ]:
# Recalibrate data
exp_data[ExperimentDataKey.UNCLASSIFIED] = recalibrate(exp_data[ExperimentDataKey.UNCLASSIFIED], Detector.ZERO)

In [ ]:
# Get PSD/Energy 2D histogram
start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

psd_report: pd.DataFrame = exp_data[ExperimentDataKey.UNCLASSIFIED]

calib_columns = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]

for calib_column, calib_key in zip(calib_columns, calib_keys):
    Z, xe, ye = get_psd_energy_histogram(
        psd_report, 
        calib_column,
        energy_width=energy_width
    )
    
    exp_data[calib_key] = {
        ExperimentDataKey.PSD_HISTOGRAM: Z,
        ExperimentDataKey.HISTOGRAM_X_EDGES: xe,
        ExperimentDataKey.HISTOGRAM_Y_EDGES: ye,
        ExperimentDataKey.END_SCAN_IDX: min(end_scan_idx, len(Z))
    }

In [ ]:
psd_report

In [ ]:
# fig, ax = plt.subplots()
# ax.hist(psd_report[DetectorDataframeColumn.RECALIBRATED_ENERGY.value], bins=100, alpha=0.2)
# # ax.set_xlim(1, 1.4)
# # ax.set_ylim(0, 20000)
# plt.hist(psd_report[DetectorDataframeColumn.CALIB_ENERGY.value], bins=100, alpha=0.2)
# # plt.hist(df[DetectorDataframeColumn.ENERGY.value], bins=100, alpha=0.2)
# plt.show()

In [ ]:
# Scan energy slices, get bimodal fits
stop_here = False

caen_calibrated_data = exp_data[ExperimentDataKey.CAEN_CALIBRATION]
new_calibrated_data = exp_data[ExperimentDataKey.NEW_CALIBRATION]

for calib_key in [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]:
    calibrated_data = exp_data[calib_key]
    Z = calibrated_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = calibrated_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = calibrated_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = calibrated_data[ExperimentDataKey.END_SCAN_IDX]
    
    # Default
    default_bounds: BimodalBounds = (
        BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
        BimodalParams(0.2, 0.1, Z.max(),
            0.38, 0.04, 4000)
    )
    
    bounds_a: BimodalBounds = (
        BimodalParams(0.1, 0.01, 1,
            0.34, 0.01, 0),
        BimodalParams(0.2, 0.1, Z.max(),
            0.36, 0.04, 4000)
    )
    
    bounds_b: BimodalBounds = (
        BimodalParams(0.1, 0.01, 1,
            0.34, 0.01, 0),
        BimodalParams(0.2, 0.1, Z.max(),
            0.36, 0.03, 4000)
    )
    
    # Ranged Example
    bounds = [
        ((0, 60), bounds_a),
    ]
    
    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, experiment_id)
    if bad_slice_indexes is not None:
        calibrated_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        calibrated_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        calibrated_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# # histogram contour plot (vaporwave island)
# cmap = plt.colormaps["nipy_spectral"]
# figsize = (24, 24)
# fontsize = 16
# histo_res = 128
# contour_res = 100
# angle_elev = 45
# angle_rot = 45

# fig = plt.figure(figsize=figsize)
# keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
# for i, calib_key in enumerate(keys):
#     data_dict = exp_data[calib_key]
#     Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
#     xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
#     ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
#     xlo = xe[:-1]
#     xhi = xe[1:]
#     ylo = ye[:-1]
#     yhi = ye[1:]
#     xmid = (xe[1:]+xe[:-1])/2
#     ymid = (ye[1:]+ye[:-1])/2
    
#     # ax = plt.axes(projection='3d')
#     ax = fig.add_subplot(len(keys), 1, i+1, projection='3d')
#     x, y = np.meshgrid(xmid, ymid)

#     ax.view_init(angle_elev, angle_rot)
#     ax.contour3D(x, y, Z.T, contour_res, cmap=cmap, alpha=0.6)
#     # ax.plot_surface(x, y, Z.T, cmap=cmap, alpha=0.6, rstride=1, cstride=10)
#     ax.contourf(x, y, Z.T, zdir='x', offset=2.6, cmap=cmap)
#     ax.contourf(x, y, Z.T, zdir='y', offset=-0.1, cmap=cmap)
#     ax.set(xlim=(-0.1, 2.6), ylim=(-0.1, 0.6))
#     # xpos = x.flatten()
#     # ypos = y.flatten()
#     # zpos = np.zeros_like(xpos)
#     # dx = xe[1]-xe[0]
#     # dy = ye[1]-ye[0]
#     # dz = Z.T.flatten()
#     # ax.bar3d(xpos, ypos, zpos, dx, dy, dz, cmap=cmap)
#     ax.set_title(f"PSD/Energy 3D Histogram - {calib_key.value}",
#                  fontsize=fontsize+4)
#     ax.set_ylabel("PSD", fontsize=fontsize)
#     ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
#     ax.set_zlabel("Counts", fontsize=fontsize)

# plt.show()

In [ ]:
# # Graph FOM threshold slice
# from data_processing.processing.figure_of_merit import bimodal
# slice_idx=30

# data_dict = exp_data[ExperimentDataKey.CAEN_CALIBRATION]
# xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
# ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
# Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
# df = data_dict[ExperimentDataKey.FOM_RESULTS]
# slice_energy = xe[slice_idx]
# ylo = ye[:-1]
# yhi = ye[1:]
# ymid = (ylo+yhi)/2
# histo_slice = Z.T[:, slice_idx]
# fom_slice = df.loc[slice_idx]
# fom = fom_slice[SliceFitDataframeColumn.FOM.value]
# params = tuple(fom_slice.iloc[1:7])

# fig, ax = plt.subplots(figsize=(8,8))
# ax.plot(ymid, histo_slice, "k", lw=4)
# ax.plot(ymid, bimodal(ylo, *params), "r--", lw=4)
# # ax.set_ylim(1, 6e3)
# ax.grid()
# # ax.set_yscale("log")
# ax.set_ylabel("Counts", fontsize=fontsize)
# ax.set_xlabel("PSD", fontsize=fontsize)
# ax.set_title(f"E={slice_energy:.3f} MeVee; FOM={fom:.3f}", fontsize=fontsize)
# ax.tick_params(axis='both', which='major', labelsize=fontsize)
# ax.tick_params(axis='both', which='minor', labelsize=fontsize)

In [ ]:
# Generate neutron window
from data_processing.types import WindowBorders
sigma = 5  # Hey y'all, here's where you change sigma!

for calib_key in [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]:
    calib_data = exp_data[calib_key]
    df = calib_data[ExperimentDataKey.FOM_RESULTS]

    classic_nasa_borders = generate_nasa_neutron_window(df)
    mod_classic_nasa = WindowBorders(left=None, right=classic_nasa_borders.right, top=classic_nasa_borders.top, bottom=classic_nasa_borders.bottom)
    recalc_nasa_borders = generate_nasa_neutron_window(df, recalculate_lower_energy_bound=True)
    new_borders = generate_n_distro_neutron_window(df, sigma=sigma)
    # calib_data[ExperimentDataKey.NASA_BORDERS] = classic_nasa_borders
    calib_data[ExperimentDataKey.NASA_BORDERS] = mod_classic_nasa
    calib_data[ExperimentDataKey.NASA_BORDERS_RECALC] = recalc_nasa_borders
    calib_data[ExperimentDataKey.N_WINDOW_BORDERS] = new_borders

In [ ]:
# classify neutrons
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
energy_column_keys = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
border_keys = [ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC, ExperimentDataKey.N_WINDOW_BORDERS]
n_column_keys = [DetectorDataframeColumn.NEUTRON_CLASS, DetectorDataframeColumn.NEUTRON_RECALC_CLASS, DetectorDataframeColumn.NEW_N_CLASS]

for calib_key, en_col_key in zip(calib_keys, energy_column_keys):
    calib_data = exp_data[calib_key]
    calib_psd_report = psd_report.copy()
    for border_key, n_col_key in zip(border_keys, n_column_keys):
        borders = calib_data[border_key]
        calib_psd_report = classify(calib_psd_report, en_col_key, borders, n_col_key)
    calib_data[ExperimentDataKey.PSD_REPORT] = calib_psd_report

In [ ]:
psd_report_caen = exp_data[ExperimentDataKey.CAEN_CALIBRATION][ExperimentDataKey.PSD_REPORT]
psd_report_new = exp_data[ExperimentDataKey.NEW_CALIBRATION][ExperimentDataKey.PSD_REPORT]
print(psd_report_caen[psd_report_caen[DetectorDataframeColumn.NEUTRON_CLASS.value] == True].shape)
print(psd_report_new[psd_report_new[DetectorDataframeColumn.NEUTRON_CLASS.value] == True].shape)

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    # prefix = np.zeros((n-1,))
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))
    # return mov_avg

def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))
# >>> a = numpy.empty((3,3,))
# >>> a[:] = numpy.nan

test_arr = np.arange(0,10)
print(moving_average(test_arr))
print(moving_average_centered(test_arr))

In [ ]:
# def neutron_energy_fn(Ep):
#     # given Ep, returns L
#     # (we actually want Ep given L, but we can curve fit for that)
#     # f = 1000 * (0.748*Ep - 2.41*(1-exp(-0.298*Ep)));
#     a = 1000
#     b = 0.748
#     c = 2.41
#     d = -0.298
#     return a * (b * Ep - c * (1 - exp(d * Ep)))

def get_light_output_converter() -> interp1d:
    interp_fn_path = Path() / "light_output_n_energy_rough_fn.pkl"
    l_ep_data_path = Path() / "l_ep_data.txt"

    try:
        with open(interp_fn_path, "rb") as fn_file:
            interpolator = pickle.load(fn_file)
    except (OSError, pickle.PickleError):
        result = np.loadtxt(l_ep_data_path)
        L, Ep = result.T
        interpolator = interp1d(L / 1000, Ep, kind="cubic")
        with open(interp_fn_path, "wb") as fn_file:
            pickle.dump(interpolator, fn_file)

    return interpolator

converter = get_light_output_converter()
converter(0.584)

In [ ]:
result = np.loadtxt(Path() / "l_ep_data.txt")
# print(np.array_str(result, precision=3, suppress_small=True))
L, Ep = result.T
# print(np.array_str(L, precision=2, suppress_small=True))
interp_fn = interp1d(L / 1000, Ep)

In [ ]:
psd_report_new = exp_data[ExperimentDataKey.NEW_CALIBRATION][ExperimentDataKey.PSD_REPORT]
g_only_new = psd_report_new[psd_report_new[DetectorDataframeColumn.NEUTRON_CLASS.value] == False]
n_only_new = psd_report_new[psd_report_new[DetectorDataframeColumn.NEUTRON_CLASS.value] == True]

# conversion function from light output (RECALIB_ENERGY) to neutron energy
# L = something(Ep)
# We have L, we need Ep
# so we need to run a solver, give it L and fn, so it outputs 
# L = fn(Ep) -> 0 = fn(Ep) - L
# root of this is Ep for given L, AKA solution!
# def neutron_energy_fn(Ep): return ...
# root_scalar(lambda x: neutron_energy_fn(x) - L, 

print(n_only_new.head())

In [ ]:
# psd_report_caen = exp_data[ExperimentDataKey.CAEN_CALIBRATION][ExperimentDataKey.PSD_REPORT]
# n_only_caen = psd_report_caen[psd_report_caen[DetectorDataframeColumn.NEUTRON_CLASS.value] == True]
bins = exp_data[ExperimentDataKey.NEW_CALIBRATION][ExperimentDataKey.HISTOGRAM_X_EDGES]

fig, ax = plt.subplots(figsize=(8, 8))
n, bins_out, *_ = ax.hist(psd_report_new[DetectorDataframeColumn.RECALIBRATED_ENERGY.value], bins=bins, alpha=0.2)
# ax.hist(g_only_new[DetectorDataframeColumn.RECALIBRATED_ENERGY.value], bins=bins, alpha=0.2)
n2, bins_out, *_ = ax.hist(n_only_new[DetectorDataframeColumn.RECALIBRATED_ENERGY.value], bins=bins, alpha=0.2)
bin_mids = (bins_out[1:]+bins_out[:-1])/2
n_moving_avg = moving_average_centered(n2)
# ax.plot(bin_mids, n_moving_avg, "o")
ax.set_xlim(0, 1.2)
# ax.set_ylim(0, 20000)
# ax.
# ax.hist(psd_report[DetectorDataframeColumn.CALIB_ENERGY.value], bins=bins, alpha=0.2)
# plt.hist(df[DetectorDataframeColumn.ENERGY.value], bins=100, alpha=0.2)
plt.show()

In [ ]:
df = pd.DataFrame(data={"Light output (MeVee)": bin_mids, "Neutron count": n2})
df

In [ ]:
bin_mids_n_energy = converter(bin_mids)
bin_edges_n_energy = converter(bins_out)
bin_lo_edge_n_energy = bin_edges_n_energy[:-1]
bin_hi_edge_n_energy = bin_edges_n_energy[1:]
bin_width_n_energy = bin_hi_edge_n_energy-bin_lo_edge_n_energy

In [ ]:
df["Energy (MeV)"] = bin_mids_n_energy
df = df[["Light output (MeVee)", "Energy (MeV)", "Neutron count"]]
df

In [ ]:
experiment_root = get_report_root(experiment_id)
exp_path = experiment_root / f"{experiment_id}_n_spectrum.csv"
df.to_csv(exp_path, index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
# ax.hist(n_only_new[DetectorDataframeColumn.RECALIBRATED_ENERGY.value], bins=bins, alpha=0.2)
# ax.bar(bin_lo_edge_n_energy, n, width=bin_width_n_energy, align="edge", alpha=0.2)
ax.bar(bin_lo_edge_n_energy, n2, width=bin_width_n_energy, align="edge", alpha=0.2)
# ax.plot(bin_mids_n_energy, n_moving_avg, "o")
plt.show()


In [ ]:
data_dict = dict(bin_mids=bin_mids, all=n, neutrons=n2, n_avg=n_moving_avg)
export_df = pd.DataFrame(data=data_dict)
# print(export_df.head(20))
csv_path = Path() / f"{experiment_id}-n_spectrum.csv"
export_df.to_csv(csv_path, header=["Energy (MeVee)", "Count (All)", "Count (Neutrons)", "Moving Average (Neutrons)"])

In [ ]:
def slope(x_arr, y_arr):
    # slope for bin based on previous and next element
    x_next = x_arr[2:]
    y_next = y_arr[2:]
    x_prev = x_arr[:-2]
    y_prev = y_arr[:-2]
    rise = np.concatenate(
        [np.array([np.nan]), y_next-y_prev, np.array([np.nan])])
    run = np.concatenate(
        [np.array([np.nan]), x_next-x_prev, np.array([np.nan])])
    slope = rise / run
    return slope

x = np.arange(0,10, dtype=float)
y = x * 2.4
slope(x, y)

In [ ]:
gradient = np.gradient(n2, bins_out[1]-bins_out[0])
# gradient = slope(bin_mids, n2)
gradient_mov_avg = np.gradient(n_moving_avg, bins_out[1]-bins_out[0])
smoothed_gradient = moving_average_centered(gradient_mov_avg, 5)
# gradient_mov_avg = slope(bin_mids, n_moving_avg)
# TODO make own gradient function and use instead of numpy gradient
# bin_mids = (bins_out[1:]+bins_out[:-1])/2

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(bin_mids, gradient, "-")
ax.plot(bin_mids, gradient_mov_avg, "--")
# ax.plot(bin_mids, smoothed_gradient, "--")

ax.set_xlim(0, 1)
ax.set_ylim(-0.2e6, 0.1e6)
plt.show()

In [ ]:
# Fit gaussian to derivative
fit_region_start_index = np.argmax(bin_mids >= 0.4)
guess = (0.584, 0.164, -60e3)
min_bounds = (0.45, 0.05, -85e3)
max_bounds = (0.70, 0.250, -40e3)
bounds = (min_bounds, max_bounds)
fit_params, _ = curve_fit(
    gaussian,
    bin_mids[fit_region_start_index:-3],
    gradient_mov_avg[fit_region_start_index:-3],
    p0=guess,
    bounds=bounds
)
print(np.array_str(fit_params, precision=4, suppress_small=True))

fit_data = gaussian(bin_mids, *fit_params)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(bin_mids, gradient_mov_avg, "--")
ax.plot(bin_mids, fit_data, "-")
ax.plot(bin_mids, n2, ".")

ax.axvline(0.584)

ax.set_xlim(0, 1)
ax.set_ylim(-0.1e6, 0.2e6)
plt.show()

In [ ]:
-gradient_mov_avg[fit_region_start_index:]

In [ ]:
mu, sigma, _ = fit_params
print(mu)
rect_min = mu-sigma
rect_max = mu+sigma
enq_res = 0.164
enq_min = mu-enq_res
enq_max = mu+enq_res

fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_mids, n2, width=bins_out[1]-bins_out[0], align="center", alpha=0.2)
# ax.plot(bin_mids, n_moving_avg, "o")

ax.axvline(mu)
ax.axvspan(rect_min, rect_max, alpha=0.2, color="m")
ax.axvspan(enq_min, enq_max, alpha=0.2, color="g")

ax.set_xlim(0, 1)
# ax.set_ylim(0, 20000)
ax.set_xlabel("Light output (MeVee)")
ax.set_ylabel("Count")

ax.annotate(f"Compton edge\n({mu:.4f} MeVee)", xy=(mu, 0.2), xycoords=("data", "axes fraction"), xytext=(10, 10), textcoords="offset points")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_lo_edge_n_energy, n2, width=bin_width_n_energy, align="edge", alpha=0.2)
mu, sigma, _ = fit_params
mu_n = converter(mu)
print(mu_n)
rect_min = converter(mu-sigma)
rect_max = converter(mu+sigma)
enq_res = 0.164
enq_min = converter(mu-enq_res)
enq_max = converter(mu+enq_res)
ax.axvline(mu_n)
ax.axvspan(rect_min, rect_max, alpha=0.2, color="m")
# ax.axvspan(enq_min, enq_max, alpha=0.2, color="g")
ax.set_xlim(1, 3.5)
ax.set_xlabel("Neutron energy (MeV)")
ax.set_ylabel("Count")
ax.annotate(f"Compton edge\n({mu_n:.4f} MeV)", xy=(mu_n, 0.2), xycoords=("data", "axes fraction"), xytext=(10, 10), textcoords="offset points")
# ax.set_ylim(0, 20000)
plt.show()

## TEMPORARY - Neutron Data Analysis

### Simpler attempt at proving Compton edge

In [ ]:
# load data from Aref (just NPS, det_pulse)
sim_data_path = Path() / "ID-419_sim_n_light_output4.txt"
sim_df = pd.read_fwf(sim_data_path)
sim_df = sim_df[["NPS", "det_pulse (MeVee)"]].copy()
sim_df.columns = ["Count Rate", "Neutron light output (MeVee)"]
sim_df.head()

In [ ]:
# TODO cut using light output bins
bins = exp_data[ExperimentDataKey.NEW_CALIBRATION][ExperimentDataKey.HISTOGRAM_X_EDGES]
light_output_cut = pd.cut(sim_df["Neutron light output (MeVee)"], bins=bins)
print(light_output_cut.head())
# binned_sim_data = sim_df.groupby(

In [ ]:
# TODO groupby cuts, using sum of NPS
binned_sim_data = sim_df.groupby(light_output_cut).sum()[["Count Rate"]].copy()
binned_sim_energy_bins = binned_sim_data.index.to_series()
midpoints = binned_sim_energy_bins.apply(lambda x: x.mid)
binned_sim_data["Bin midpoint"] = midpoints
print(binned_sim_data.head())

In [ ]:
# TODO plot with experimental data
mu, sigma, _ = fit_params
print(mu)
rect_min = mu-sigma
rect_max = mu+sigma
enq_res = 0.164
enq_min = mu-enq_res
enq_max = mu+enq_res
bin_width = bins_out[1]-bins_out[0]

# normalization
n2_max = max(n2)
sim_counts = binned_sim_data["Count Rate"]
sim_max = max(sim_counts)
n2_norm = n2 / n2_max
sim_norm = sim_counts / sim_max

fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_mids, n2_norm, width=bin_width, align="center", alpha=0.2)
ax.bar(bin_mids, sim_norm, width=bin_width, align="center", alpha=0.2)
# ax.plot(bin_mids, n_moving_avg, "o")

# ax.axvline(mu)
# ax.axvspan(rect_min, rect_max, alpha=0.2, color="m")
# ax.axvspan(enq_min, enq_max, alpha=0.2, color="g")

ax.set_xlim(0, 0.9)
# ax.set_ylim(0, 20000)
ax.set_xlabel("Light output (MeVee)")
ax.set_ylabel("Normalized Count (a.u.)")

# ax.annotate(f"Compton edge\n({mu:.4f} MeVee)", xy=(mu, 0.2), xycoords=("data", "axes fraction"), xytext=(10, 10), textcoords="offset points")

plt.show()

In [ ]:
print(np.array_str(bin_width_n_energy, precision=3, suppress_small=True))

In [ ]:
bin_width_n_energy.shape

In [ ]:
mu, sigma, _ = fit_params
mu_n = converter(mu)
print(mu_n)
print(converter(sigma))
rect_min = converter(mu-sigma)
rect_max = converter(mu+sigma)
enq_res = 0.164
enq_min = converter(mu-enq_res)
enq_max = converter(mu+enq_res)

sim_fit_data = gaussian(bin_lo_edge_n_energy, 2.45, 0.2, 0.23)

fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_lo_edge_n_energy, n2_norm, width=bin_width_n_energy, align="edge", alpha=0.2)
ax.bar(bin_lo_edge_n_energy, sim_norm, width=bin_width_n_energy, align="edge", alpha=0.2)
# ax.plot(bin_lo_edge_n_energy, sim_fit_data, "-")

# ax.axvline(mu_n)
ax.axvline(2.45)
# ax.axvspan(rect_min, rect_max, alpha=0.2, color="m")
# ax.axvspan(enq_min, enq_max, alpha=0.2, color="g")

ax.set_xlim(1.29, 3.25)
# ax.set_ylim(0, 20000)
ax.set_xlabel("Neutron energy (MeV)")
ax.set_ylabel("Normalized count (a.u)")

# ax.annotate(f"Compton edge\n({mu_n:.4f} MeV)", xy=(mu_n, 0.2), xycoords=("data", "axes fraction"), xytext=(10, 10), textcoords="offset points")

plt.show()

### Convolution attempt

In [ ]:
bin_mids

In [ ]:
def detector_resolution(E, a=0.102, b=0.102, c=0.036):
    term_one = np.square(a)
    term_two = np.square(b)/E
    term_three = np.square(c/E)
    resolution = np.sqrt(term_one+term_two+term_three) * E
    return resolution

In [ ]:
filter_x = bin_mids[:40]
res_filter = detector_resolution(filter_x)
filtered = convolve(sim_counts, filter_x, mode='same')

In [ ]:
filtered_max = max(filtered)
filtered_norm = filtered / filtered_max

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_mids, n2_norm, width=bin_width, align="center", alpha=0.2)
# ax.bar(bin_mids, sim_norm, width=bin_width, align="center", alpha=0.2)
ax.bar(bin_mids, filtered_norm, width=bin_width, align="center", alpha=0.2)
# ax.plot(bin_mids, n_moving_avg, "o")

# ax.axvline(mu)
# ax.axvspan(rect_min, rect_max, alpha=0.2, color="m")
# ax.axvspan(enq_min, enq_max, alpha=0.2, color="g")

ax.set_xlim(0.2, 0.9)
# ax.set_ylim(0, 20000)
ax.set_xlabel("Light output (MeVee)")
ax.set_ylabel("Normalized Count (a.u.)")

# ax.annotate(f"Compton edge\n({mu:.4f} MeVee)", xy=(mu, 0.2), xycoords=("data", "axes fraction"), xytext=(10, 10), textcoords="offset points")

plt.show()

### New section

In [ ]:
# deconvolution
# inspired by https://stackoverflow.com/a/40656281
# do gaussian fit on neutron pulse height distribution
# mu is from prior fit, sigma should be 0.164 MeVee

# phd_sigma = 0.164
# fit_region_start_index = np.argmax(bin_mids >= mu)
# print(fit_region_start_index)
# phd_gaussian = lambda x, a: gaussian(x, mu, phd_sigma, a)
# guess = (10e3)
# min_bounds = (5e3)
# max_bounds = (15e3)
# bounds = (min_bounds, max_bounds)
# fit_params, _ = curve_fit(
#     phd_gaussian,
#     bin_mids[fit_region_start_index:],
#     n2[fit_region_start_index:],
#     p0=guess,
#     bounds=bounds
# )
# print(np.array_str(fit_params, precision=4, suppress_small=True))

# didn't work! Now trying to just limit mu to 0.584 M and get sigma/a via fit
mu_2 = 0.584
# phd_gaussian_test = lambda x, sigma, a: gaussian(x, mu, sigma, a)
phd_gaussian_test = lambda x, sigma, a: gaussian(x, mu_2, sigma, a)
guess = (0.164, 15e3)
min_bounds = (0.05, 5e3)
max_bounds = (0.25, 20e3)
bounds = (min_bounds, max_bounds)
fit_params_test, _ = curve_fit(
    phd_gaussian_test,
    bin_mids[fit_region_start_index:],
    n2[fit_region_start_index:],
    p0=guess,
    bounds=bounds
)
print(np.array_str(fit_params_test, precision=4, suppress_small=True))

# fit_data = phd_gaussian(bin_mids, *fit_params)
fit_data_test = phd_gaussian_test(bin_mids, *fit_params_test)
fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_mids, n2, width=bins_out[1]-bins_out[0], align="center", alpha=0.2)
# ax.plot(bin_mids, fit_data, "-")
ax.plot(bin_mids, fit_data_test, "-")
ax.set_xlim(0, 1)

# # TODO create gaussian using fit_params and x (linspace(0.4, 1))
filter_x = np.linspace(0.3, 1)
gauss_filter = phd_gaussian_test(filter_x, *fit_params_test)

# apply gaussian in deconvolve fn
deconv, _ = deconvolve(n2, gauss_filter)

# correct size of deconvolution array
n = len(n2) - len(gauss_filter) + 1
s = (len(n2) - n) // 2
deconv_res = np.zeros(len(n2))
deconv_res[s:len(n2)-s-1] = deconv
deconv = deconv_res

# # TODO convert values to N energy

# TODO plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_mids, deconv, width=bins_out[1]-bins_out[0], align="center", alpha=0.2)
plt.show()

## Export and Display

In [ ]:
# Plot classification
HISTOGRAM_RES = 1024
COUNT_LIMIT = 20

calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
energy_column_keys = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
border_keys = [
    ExperimentDataKey.NASA_BORDERS,
    ExperimentDataKey.NASA_BORDERS_RECALC,
    ExperimentDataKey.N_WINDOW_BORDERS
]
n_column_keys = [
    DetectorDataframeColumn.NEUTRON_CLASS, 
    DetectorDataframeColumn.NEUTRON_RECALC_CLASS, 
    DetectorDataframeColumn.NEW_N_CLASS
]

for calib_key, en_col_key in zip(calib_keys, energy_column_keys):
    calib_data = exp_data[calib_key]
    psd_report = calib_data[ExperimentDataKey.PSD_REPORT]
    for border_key, n_col_key in zip(border_keys, n_column_keys):
        borders = calib_data[border_key]
        print(borders)
    
        fig, ax = plot_classification(
            psd_report,
            borders,
            experiment_id,
            n_col_key,
            en_col_key,
            count_limit=COUNT_LIMIT,
            colormap_name="seismic"
        )

        print(f"{calib_key.value}/{border_key.value}")
        print(borders.left)
        plt.show()

In [ ]:
# Save window boundaries
save_folder = INPUT_DATA_FOLDER / "ReferenceWindow"
save_folder.mkdir(parents=True, exist_ok=True)

calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
energy_column_keys = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
border_keys = [
    ExperimentDataKey.NASA_BORDERS,
    ExperimentDataKey.NASA_BORDERS_RECALC,
    ExperimentDataKey.N_WINDOW_BORDERS
]
n_column_keys = [
    DetectorDataframeColumn.NEUTRON_CLASS, 
    DetectorDataframeColumn.NEUTRON_RECALC_CLASS, 
    DetectorDataframeColumn.NEW_N_CLASS
]

for calib_key in calib_keys:
    calib_data = exp_data[calib_key]
    for border_key in border_keys:
        borders = calib_data[border_key]
    
        left_border = borders.left
        right_border = borders.right
        # TODO put calibration type in save file path
        side_borders_file_path = save_folder / f"{calib_key.value}_{border_key.value}_side_borders.txt"
        with side_borders_file_path.open('w') as side_file:
            side_file.writelines(
                [
                    f"left: {left_border if left_border is not None else 'None'}\n",
                    f"right: {right_border if right_border is not None else 'None'}\n"
                ]
            )
        print(f"Saved side borders to {side_borders_file_path}")
        
        bottom_border = borders.bottom
        if bottom_border is not None:
            bottom_border_file_path = save_folder / f"{calib_key.value}_{border_key.value}_bottom_border.pkl"
            with bottom_border_file_path.open('wb') as bottom_file:
                pickle.dump(bottom_border, bottom_file)
            print(f"Saved bottom border to {bottom_border_file_path}")
        else:
            print("No bottom border")
    
        top_border = borders.top
        if top_border is not None:
            top_border_file_path = save_folder / f"{calib_key.value}_{border_key.value}_top_border.pkl"
            with top_border_file_path.open('wb') as top_file:
                pickle.dump(top_border, top_file)
            print(f"Saved top border to {top_border_file_path}")
        else:
            print("No top border")

In [ ]:
input("Processing done, hit Enter to finish")
stop()